# dbt (Data Build Tool)
**Day 10 — Technology & Architecture Module**

---

<div style="padding:15px;border-left:8px solid #667eea;background:#f0f0ff;border-radius:4px;">
<strong>Core Insight:</strong> dbt is to SQL what git is to code. It adds version control, testing, documentation, and lineage to SQL transformations. Instead of untracked SQL scripts running in Redshift or Snowflake, dbt gives you a software engineering workflow for your data models. It's now the standard in modern data engineering — if you haven't used it, you need to know the concepts.
</div>

### The dbt Mental Model
```text
Source data (raw tables in warehouse)
     ↓
  dbt models (SELECT statements in .sql files)
     ↓
  dbt tests (assertions about your data)
     ↓
  Compiled tables/views in the warehouse
     ↓
  Documentation + lineage graph auto-generated
```

## Code as Configuration: `ref()` and Materializations

A dbt "model" is simply a `SELECT` statement in a `.sql` file. The magic comes from the `ref()` function and `{{ config(...) }}` blocks.

### Dependency Resolution Tree
```text
sources.yml (raw source definitions)
  ├──→ stg_users.sql (View)
  ├──→ stg_orders.sql (View)
         └──→ fct_orders.sql (Table) ← uses {{ ref('stg_orders') }}
                └──→ mart_monthly_revenue.sql (Incremental Table)
```

### Materialization Tradeoffs
| Materialization | How it builds in Warehouse | Best For |
|-----------------|----------------------------|----------|
| `view` | `CREATE VIEW AS ...` | Staging layers, lightweight logic, always fresh |
| `table` | `CREATE TABLE AS ...` | Intermediate transformations, slow BI queries |
| `incremental` | Insert/Merge new rows only | Massive event tables (logs, telemetry) |
| `ephemeral` | Injected as CTEs | Avoiding cluttering the warehouse with views |

In [ ]:
-- Production dbt Code: A staging model and a mart model
-- (This represents what sits in your dbt repository's SQL files)

-- File: models/staging/stg_server_metrics.sql
{{ config(materialized='view') }}

SELECT
    server_id,
    report_date::DATE AS report_date,
    avg_cpu::FLOAT AS avg_cpu,
    UPPER(region) AS region,
    LOWER(tier) AS tier
FROM {{ source('raw', 'daily_metrics') }}   -- references raw source table
WHERE avg_cpu IS NOT NULL
  AND report_date >= '2026-01-01';


-- File: models/marts/server_daily_summary.sql
{{ config(
    materialized='table',
    partition_by={'field': 'report_date', 'data_type': 'date'}
) }}

WITH base AS (
    SELECT * FROM {{ ref('stg_server_metrics') }}   -- ref() handles DAG ordering
),
with_classification AS (
    SELECT
        *,
        CASE
            WHEN avg_cpu >= 90 THEN 'critical'
            WHEN avg_cpu >= 75 THEN 'warning'
            ELSE 'normal'
        END AS cpu_status
    FROM base
)
SELECT
    server_id,
    report_date,
    ROUND(AVG(avg_cpu), 2) AS avg_cpu,
    MAX(avg_cpu) AS peak_cpu,
    MAX(cpu_status) AS worst_status
FROM with_classification
GROUP BY server_id, report_date;


In [ ]:
# LOCAL SIMULATION — Simulate dbt's Jinja Compilation & Execution
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
    CREATE TABLE raw_daily_metrics (server_id TEXT, report_date TEXT, avg_cpu REAL, region TEXT, tier TEXT);
    INSERT INTO raw_daily_metrics VALUES ('srv-01', '2026-02-27', 72.5, 'us-east', 'silver');
    INSERT INTO raw_daily_metrics VALUES ('srv-02', '2026-02-27', 91.3, 'us-west', 'gold');
""")

# Simulate compiled `stg_server_metrics.sql`
print("DBT SIMULATION: Running staging model (View creation)")
conn.execute("""
    CREATE VIEW stg_server_metrics AS 
    SELECT server_id, report_date, avg_cpu, UPPER(region) AS region, LOWER(tier) AS tier
    FROM raw_daily_metrics WHERE avg_cpu IS NOT NULL
""")

# Simulate compiled `server_daily_summary.sql`
print("DBT SIMULATION: Running mart model (Table creation)\n")
conn.execute("""
    CREATE TABLE server_daily_summary AS
    WITH base AS ( SELECT * FROM stg_server_metrics ),
    with_classification AS (
        SELECT *, CASE WHEN avg_cpu >= 90 THEN 'critical'
                       WHEN avg_cpu >= 75 THEN 'warning'
                       ELSE 'normal' END AS cpu_status FROM base
    )
    SELECT server_id, report_date, ROUND(AVG(avg_cpu), 2) AS avg_cpu, 
           MAX(avg_cpu) AS peak_cpu, MAX(cpu_status) AS worst_status
    FROM with_classification GROUP BY server_id, report_date;
""")

print("Querying generated 'marts.server_daily_summary':")
print(pd.read_sql_query("SELECT * FROM server_daily_summary", conn).to_string())


DBT SIMULATION: Running staging model (View creation)
DBT SIMULATION: Running mart model (Table creation)

Querying generated 'marts.server_daily_summary':
  server_id report_date  avg_cpu  peak_cpu worst_status
0    srv-01  2026-02-27    72.50      72.5       normal
1    srv-02  2026-02-27    91.30      91.3     critical


## Data Quality as Code (Schema Tests)

dbt prevents bad data from failing downstream systems by letting you define SQL-based assertions using simple YAML configuration files. If tests fail on source data, dbt can stop the transform pipeline immediately.

In [ ]:
# Production dbt Testing (YAML configuration)
# File: models/staging/schema.yml
"""
version: 2

models:
  - name: stg_server_metrics
    columns:
      - name: server_id
        tests:
          - not_null
          - unique           # each server_id+date combination should be unique (handled below)
      - name: avg_cpu
        tests:
          - not_null
          - dbt_utils.accepted_range:
              min_value: 0
              max_value: 100
      - name: region
        tests:
          - accepted_values:
              values: ['US-EAST', 'US-WEST', 'EU-WEST']

  - name: server_daily_summary
    tests:
      - dbt_utils.unique_combination_of_columns:
          combination_of_columns: [server_id, report_date]
"""

In [ ]:
# LOCAL SIMULATION — How dbt complies tests into SQL under the hood

conn.execute("INSERT INTO raw_daily_metrics VALUES ('srv-error', '2026-02-27', 150.0, 'us-east', 'bronze');")

print("Simulating dbt test: not_null on server_id")
res = conn.execute("SELECT COUNT(*) FROM raw_daily_metrics WHERE server_id IS NULL").fetchone()[0]
if res == 0:
    print("  ✅ PASS: 0 null records found.")

print("\nSimulating dbt test: accepted_range (0-100) on avg_cpu")
res = conn.execute("SELECT COUNT(*) FROM raw_daily_metrics WHERE avg_cpu < 0 OR avg_cpu > 100").fetchone()[0]
if res > 0:
    print(f"  ❌ FAIL: Found {res} records violating range (e.g. cpu=150.0)")


Simulating dbt test: not_null on server_id
  ✅ PASS: 0 null records found.

Simulating dbt test: accepted_range (0-100) on avg_cpu
  ❌ FAIL: Found 1 records violating range (e.g. cpu=150.0)


## Incremental Execution & Orchestration

For massive tables, a full `CREATE TABLE AS` rebuild every 24 hours is prohibitively expensive. dbt uses **incremental models** leveraging jinja `{% if is_incremental() %}` blocks to only process the raw rows that arrived since the last run.

### Cost Math (BigQuery / Snowflake DWH Run)
| Strategy | Data Scanned Daily | Daily Run Cost | Monthly DWH Cost |
|----------|-------------------|----------------|-----------------|
| Full Table Rebuild | 10 TB (all history) | $50.00 | $1,500 |
| Incremental Updates | 100 GB (new data only) | $0.50 | $15 |

You orchestrate these runs using an orchestrator like **Apache Airflow**. Airflow triggers the extraction, writes raw data to Snowflake, then kicks off a `dbt run --select staging+` execution to compile and run the downstream logic.

In [ ]:
# LOCAL SIMULATION — Incremental execution
# This simulates how dbt compiles `is_incremental()` logic

conn.executescript("""
    CREATE TABLE IF NOT EXISTS mart_events (event_id TEXT, ts TEXT);
    INSERT INTO mart_events VALUES ('e1', '2026-02-26');
    INSERT INTO mart_events VALUES ('e2', '2026-02-26');
""")
print(f"Initial full run row count: {conn.execute('SELECT COUNT(*) FROM mart_events').fetchone()[0]}")

# A new event lands in raw
conn.executescript("CREATE TABLE raw_events (event_id TEXT, ts TEXT);
                 INSERT INTO raw_events VALUES ('e3', '2026-02-27');")

# How dbt runs the incremental block: SELECT * WHERE ts > (SELECT MAX(ts) FROM destination)
max_ts = conn.execute("SELECT MAX(ts) FROM mart_events").fetchone()[0]
conn.execute(f"""
    INSERT INTO mart_events 
    SELECT * FROM raw_events WHERE ts > '{max_ts}'
""")
print("Incremental run logic simulated.")
print(f"After incremental run row count: {conn.execute('SELECT COUNT(*) FROM mart_events').fetchone()[0]}")


Initial full run row count: 2
Incremental run logic simulated.
After incremental run row count: 3


## Full Architecture Overview: Airflow + dbt + Snowflake

```text
Third-Party APIs / Databases
│
├──→ Fivetran / Airbyte (EL part of ELT)
│
▼
Snowflake / Redshift / BigQuery (Raw Data)
│
├──→ dbt Core / Cloud (The T in ELT)
│      ├──→ Source Freshness Checks
│      ├──→ Staging Views (Type Casts/Renames)
│      ├──→ Data Tests (Nulls/Uniques)
│      └──→ Mart Tables (Aggregations/Joins)
│
└──→ Orchestrated by Apache Airflow
       └──→ Triggers EL pipelines
       └──→ Triggers `dbt run` immediately after load
       └──→ Sends Slack alerts immediately if a dbt test fails
```
**Governance / Security Note:** Using standard CI/CD with dbt implies that NO developer can push raw SQL strings into the production Data Warehouse. Code has to be PR'd into the dbt GitHub repo, reviewed, and test-compiled against a clone schema before the CD pipeline is allowed to materialize the production tables. Access is natively restricted by `profiles.yml` utilizing scoped service accounts.

## Interview Q&A

**Q: What is `ref()` in dbt and why is it important?**<br>
A: `ref('model_name')` logically links one dbt model to another. Under the hood, dbt uses these `ref()` calls to automatically generate a Directed Acyclic Graph (DAG) for correct build ordering. It also dynamically resolves `ref()` to point to your specific environment schema (like `dev_sean` instead of `prod`).

**Q: What is materialization in dbt?**<br>
A: It dictates how dbt builds the model in the physical warehouse: `view` (a stored query wrapper, executes upstream every read), `table` (full CTE rebuild every execution), `incremental` (processes only net-new rows for immense tables), and `ephemeral` (injected inline purely as a CTE without database manifestation).

**Q: What are dbt sources?**<br>
A: `source()` references raw, dirty incoming tables that aren't constructed or controlled by dbt (like API drops loaded by Fivetran). Unlike `ref()`, mapping `source()` grants the ability to attach "freshness tests" ensuring your pipeline alarms if raw data stalls entirely before triggering.

**Q: How does dbt handle incremental models?**<br>
A: Utilizing the `is_incremental()` Jinja macro! On the literal first execution, dbt builds a full monolithic table. On subsequent runs, it injects a specific `WHERE event_time > (select max(event_time) from target)` filter, allowing the model to surgically append or merge new partitions instead of performing a $5,000 full historical sweep.

**Q: Design a pipeline to extract financial reporting data from 5 disjointed databases into Snowflake and expose aggregated business dashboards completely error-free.**<br>
A: I would leverage an ELT orchestration using Airflow. Airflow would trigger Airbyte to execute parallel extracts from the 5 SQL DBs and natively Load the data indiscriminately into Snowflake `RAW` schemas. Airflow then immediately triggers `dbt run`. The dbt DAG executes stringent YAML schema testing directly on the source arrays; if they clear, it compiles down `stg_views` to standardise fields, followed by `incremental_marts` for aggregation. Dashboards sit purely on the stable `marts`, insulated from raw chaos.

## The Citi Angle

**Topic: Data Quality + Automation**

- **Before:** Legacy telemetry SQL scripts lacked dependency logic and ran serially manually (or via bloated cronjobs), passing corrupted future-dated timestamps straight into executive dashboards without warning.
- **After:** Implementing an ELT paradigm enforcing schema validations intercepted the anomaly data prior to aggregation, keeping dashboards pristine and alerting owners securely via the dead-letter queue.

*"One of the persistent problems at Citi was data quality in the telemetry pipeline. Agents would report metrics with wrong timestamps — sometimes future-dated by hours due to NTP drift. This would corrupt rolling averages and make dashboards show erratic capacity trends. I built a validation layer that checked timestamp sanity (reject if more than 30 minutes in the future or more than 24 hours old) and server ID validity against our inventory. What I'd do today with dbt: add schema tests on the raw source and custom data tests for timestamp bounds — catch it at the staging layer before it reaches the mart. The dead-letter queue would hold rejects for manual review."*